In [2]:
!nvidia-smi


Wed Mar 11 04:29:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!python3 -m pip install segmentation_models_pytorch -q
!python3 -m pip install albumentations
!python3 -m pip install torchsummary torchmetrics
!python3 -m pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 18.5 MB/s eta 0:00:00


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
sys.path.append("/content/drive/MyDrive/Building-extraction-from-UAV-dev")
import random
import numpy as np # Import numpy
import segmentation_models_pytorch as smp
import torch.nn.functional
import torchvision
import math
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from tqdm import tqdm
import torchmetrics

from model.block import *
from model.encoder import *
from model.seg_model import *
from model.seg_humg_model import *
from utils.dataset_split import *
from opt_humg import *


In [6]:

"""
    block
    * conv/upsample/downsample layer
    * psp
"""

# DoubleCov Layer： (conv-> batch_norm -> ReLU)*2
class DoubleConv (nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels,
                      kernel_size=3, padding=1, stride=1, bias=False),  # zero padding
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),   # inplace: use original memory

            nn.Conv2d(out_channels, out_channels,
                       kernel_size=3, padding=1, stride=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


# DownSample Layer: ( maxpool -> DoubleCov Layer)
class DownSample (nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.down = nn.Sequential(
            nn.MaxPool2d(kernel_size=2, padding=0, stride=2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.down(x)


# UpSample Layer：( Upsample -> cat -> DoubleCov Layer)
class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.high_level = nn.Sequential(
            nn.Upsample(mode='bilinear', scale_factor=2, align_corners=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        )
        self.doubleCov = DoubleConv(in_channels,out_channels)

    def forward(self, low_level_x, high_level_x):
        # tensor: N C H W  , make sure other dimensions (H W) are the same
        # hx : (C/2, 2*H , 2*W)
        high_level_x = self.high_level(high_level_x)
        # cat (in C dim)
        cat_x = torch.cat([low_level_x, high_level_x], dim=1)
        # Double Conv
        x = self.doubleCov(cat_x)
        return x


# outConv Layer: 1*1 conv
class outConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.outconv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)

    def forward(self, x):
        return self.outconv(x)


class U_net(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_net, self).__init__()   # inherit nn.Module class
        self.classes = classes
        self.in_channels = in_channels

        # encoding
        self.Cov = DoubleConv(self.in_channels, 64)
        self.down1 = DownSample(64,128)
        self.down2 = DownSample(128, 256)
        self.down3 = DownSample(256, 512)
        self.down4 = DownSample(512, 1024)

        # decoding
        self.up1 = UpSample(1024,512)
        self.up2 = UpSample(512,256)
        self.up3 = UpSample(256,128)
        self.up4 = UpSample(128,64)

        # head
        self.out = outConv(64, self.classes)

    def forward(self, x):

        x1 = self.Cov(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x6 = self.up1(x4, x5)
        x7 = self.up2(x3, x6)
        x8 = self.up3(x2, x7)
        x9 = self.up4(x1, x8)

        x10 = self.out(x9)

        return x10


# PSP
class PSPool(nn.Module):
    def __init__(self, in_channel):
        super(PSPool, self).__init__()
        self.in_channel = in_channel

        self.pool1 = nn.MaxPool2d(kernel_size=1,stride=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2,stride=2)
        self.pool3 = nn.MaxPool2d(kernel_size=4,stride=4)
        self.pool4 = nn.MaxPool2d(kernel_size=8,stride=8)

        self.conv = nn.Conv2d(kernel_size=1,stride=1,
                              in_channels=in_channel,
                              out_channels=in_channel//4)

        self.upsample1 = nn.Upsample(mode='bilinear',scale_factor=1,align_corners=True)
        self.upsample2 = nn.Upsample(mode='bilinear',scale_factor=2,align_corners=True)
        self.upsample3 = nn.Upsample(mode='bilinear',scale_factor=4,align_corners=True)
        self.upsample4 = nn.Upsample(mode='bilinear',scale_factor=8,align_corners=True)

        self.DR = nn.Conv2d(kernel_size = 1, stride = 1,
                       in_channels = self.in_channel * 2,
                       out_channels = self.in_channel)
        self.pool5 = nn.MaxPool2d(kernel_size=2,stride=2)

    def forward(self, x):
        x1 = self.pool1(x)
        x2 = self.pool2(x)
        x3 = self.pool3(x)
        x4 = self.pool4(x)

        x1 = self.conv(x1)
        x2 = self.conv(x2)
        x3 = self.conv(x3)
        x4 = self.conv(x4)

        x1 = self.upsample1(x1)
        x2 = self.upsample2(x2)
        x3 = self.upsample3(x3)
        x4 = self.upsample4(x4)

        x5 = torch.cat([x, x1, x2, x3, x4],dim=1)

        # DR
        x6 = self.DR(x5)

        x7 = self.pool5(x6)

        return x7


In [7]:
"""
    encoder
    * resnet50
"""

class Bottleneck(nn.Module):
    def __init__(self, in_channel, out_channel, stride):
        super(Bottleneck, self).__init__()

        factor = 2

        self.conv1 = nn.Conv2d(kernel_size=1, stride=1, in_channels=in_channel, out_channels=out_channel)
        self.conv2 = nn.Conv2d(kernel_size=3, stride=stride, padding=1, in_channels=out_channel, out_channels=out_channel)
        self.conv3 = nn.Conv2d(kernel_size=1, stride=1, in_channels=out_channel, out_channels=out_channel*factor)
        self.conv_x = nn.Conv2d(kernel_size=1, stride=stride, in_channels=in_channel, out_channels=out_channel*factor)

        self.lay_1 = nn.Sequential(self.conv1, nn.BatchNorm2d(out_channel), nn.ReLU(inplace=True))
        self.lay_2 = nn.Sequential(self.conv2, nn.BatchNorm2d(out_channel), nn.ReLU(inplace=True))
        self.lay_3 = nn.Sequential(self.conv3, nn.BatchNorm2d(out_channel*factor), nn.ReLU(inplace=True))
        self.lay_x = nn.Sequential(self.conv_x, nn.BatchNorm2d(out_channel*factor))

    def forward(self, x):

        x_add = self.lay_x(x)

        x = self.lay_1(x)
        x = self.lay_2(x)
        x = self.lay_3(x)

        x1 = torch.add(input=x, alpha=1, other=x_add)
        x1 = nn.ReLU(inplace=True)(x1)

        return x1

class ResNet50Encoder(nn.Module):
    def __init__(self, in_channel):
        super(ResNet50Encoder, self).__init__()

         # stage 0
        self.stage0 = nn.Sequential(
            nn.Conv2d(in_channels=in_channel, out_channels=64,
                      kernel_size=3, stride=1, padding=1),  # 7*7 to 3*3
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1) # (256,256)->(128,128)

        # stage 1
        self.stage1_1 = Bottleneck(in_channel=64, out_channel=64, stride=1)    # 64,64,64,128
        self.stage1_2 = Bottleneck(in_channel=128, out_channel=64, stride=1)   # 128,64,64,128
        self.stage1_3 = Bottleneck(in_channel=128, out_channel=64, stride=1)   # 128,64,64,128

        # stage 2
        self.stage2_1 = Bottleneck(in_channel=128, out_channel=128, stride=2)  # 128,128,128,256 (128,128)->(64,64)
        self.stage2_2 = Bottleneck(in_channel=256, out_channel=128, stride=1)  # 256,128,128,256
        self.stage2_3 = Bottleneck(in_channel=256, out_channel=128, stride=1)  # 256,128,128,256
        self.stage2_4 = Bottleneck(in_channel=256, out_channel=128, stride=1)  # 256,128,128,256

        # stage 3
        self.stage3_1 = Bottleneck(in_channel=256, out_channel=256, stride=2)  # 256,256,256,512 (64,64)->(32,32)
        self.stage3_2 = Bottleneck(in_channel=512, out_channel=256, stride=1)  # 512,256,256,512
        self.stage3_3 = Bottleneck(in_channel=512, out_channel=256, stride=1)  # 512,256,256,512
        self.stage3_4 = Bottleneck(in_channel=512, out_channel=256, stride=1)  # 512,256,256,512
        self.stage3_5 = Bottleneck(in_channel=512, out_channel=256, stride=1)  # 512,256,256,512
        self.stage3_6 = Bottleneck(in_channel=512, out_channel=256, stride=1)  # 512,256,256,512
        self.stage3_7 = Bottleneck(in_channel=512, out_channel=256, stride=1)  # 512,256,256,512

        # stage 4
        self.stage4_1 = Bottleneck(in_channel=512, out_channel=512, stride=2)  # 512,512,512,1024 (32,32)->(16,16)
        self.stage4_2 = Bottleneck(in_channel=1024, out_channel=512, stride=1)  # 1024,512,512,1024
        self.stage4_3 = Bottleneck(in_channel=1024, out_channel=512, stride=1)  # 1024,512,512,1024

    def forward(self, x):

        x = self.stage0(x)
        x1 = x  # (256,256,64)

        x = self.pool(x)

        x = self.stage1_1(x)
        x = self.stage1_2(x)
        x = self.stage1_3(x)
        x2 = x  # (128,128,128)

        x = self.stage2_1(x)
        x = self.stage2_2(x)
        x = self.stage2_3(x)
        x = self.stage2_4(x)
        x3 = x  # (64,64,256)

        x = self.stage3_1(x)
        x = self.stage3_2(x)
        x = self.stage3_3(x)
        x = self.stage3_4(x)
        x = self.stage3_5(x)
        x = self.stage3_6(x)
        x4 = x  # (32,32,512)

        x = self.stage4_1(x)
        x = self.stage4_2(x)
        x = self.stage4_3(x)
        x5 = x  # (16,16,1024)

        return [x1, x2, x3, x4, x5]

In [8]:
# Define encoder and weights
ENCODER = 'se_resnext50_32x4d'
ENCODER_WEIGHTS = 'imagenet'

# Create segmentation model using SMP with a pretrained encoder
class U_rnet(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_rnet, self).__init__()  # inherit nn.Module class

        # SMP Unet with se_resnext50_32x4d as the encoder
        self.model = smp.Unet(
            encoder_name=ENCODER,                # Encoder: se_resnext50_32x4d
            encoder_weights=ENCODER_WEIGHTS,     # Pretrained on ImageNet
            in_channels=in_channels,             # Number of input channels
            classes=classes                      # Number of output classes
        )

    def forward(self, x):
        return self.model(x)

ENCODER = 'swin_transformer'  # Using the Swin Transformer encoder
ENCODER_WEIGHTS = 'imagenet'  # Pre-trained weights on ImageNet

# Create segmentation model using SMP with a pretrained encoder
class U_stnet(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_stnet, self).__init__()  # inherit nn.Module class

        # SMP Unet with se_resnext50_32x4d as the encoder
        self.model = smp.Unet(
            encoder_name=ENCODER,                # Encoder: se_resnext50_32x4d
            encoder_weights=ENCODER_WEIGHTS,     # Pretrained on ImageNet
            in_channels=in_channels,             # Number of input channels
            classes=classes                      # Number of output classes
        )

    def forward(self, x):
        return self.model(x)

# Define Encoder and Pretrained Weights
# ENCODER = 'swin_transformer'
ENCODER = 'timm-efficientnet-b7'  # Using EfficientNet-b7 encoder
ENCODER = 'se_resnext50_32x4d'
ENCODER_WEIGHTS = 'imagenet'

# Wrapper class for the segmentation model
class SegmentationModel(nn.Module):
    def __init__(self, architecture, in_channels, classes):
        super(SegmentationModel, self).__init__()

        # Choose the architecture dynamically
        if architecture == 'Unet':
            self.model = smp.Unet(
                encoder_name=ENCODER,
                encoder_weights=ENCODER_WEIGHTS,
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'DeepLabV3+':
            self.model = smp.DeepLabV3Plus(
                encoder_name=ENCODER,
                encoder_weights=ENCODER_WEIGHTS,
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'FPN':
            self.model = smp.FPN(
                encoder_name=ENCODER,
                encoder_weights=ENCODER_WEIGHTS,
                in_channels=in_channels,
                classes=classes
            )
        else:
            raise ValueError(f"Unknown architecture: {architecture}")

    def forward(self, x):
        return self.model(x)


# Define the Residual Convolution Block with Atrous Convolutions
class ResConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation=1, dropout_rate=0.3):
        super(ResConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_rate)  # Dropout layer
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)  # Apply dropout
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        return out

# Define the CBAM Attention Blocks
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

# Define the Convolutional Block with Attention
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, dilation=1, dropout_rate=0.3):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_rate)  # Dropout layer
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.ca = ChannelAttention(out_c)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

# Define the Encoder and Decoder Blocks
class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c, dilation=1):
        super(EncoderBlock, self).__init__()
        self.conv = ConvBlock(in_c, out_c, dilation)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.conv(x)
        p = self.pool(x)
        return x, p

class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super(DecoderBlock, self).__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_c * 2, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        x = self._pad_to_match(x, skip)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

    def _pad_to_match(self, x, skip):
        """Pad x to match the spatial dimensions of skip."""
        if x.size(2) < skip.size(2):
            pad_h = skip.size(2) - x.size(2)
            pad_w = skip.size(3) - x.size(3)
            x = F.pad(x, (0, pad_w, 0, pad_h))
        return x

# Define the ResCBAM_UNet Model
class ResCBAM_UNet_v4(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(ResCBAM_UNet_v4, self).__init__()

        # Encoder
        self.e1 = EncoderBlock(3, 64)  # 3 channels for RGB input
        self.e2 = EncoderBlock(64, 128)
        self.e3 = EncoderBlock(128, 256)
        self.e4 = EncoderBlock(256, 512)

        # Bottleneck with Atrous Convolutions
        self.b1 = ResConvBlock(512, 640, dilation=2, dropout_rate=dropout_rate)
        self.b2 = ResConvBlock(640, 768, dilation=4, dropout_rate=dropout_rate)
        self.b3 = ResConvBlock(768, 896, dilation=8, dropout_rate=dropout_rate)
        self.b4 = ResConvBlock(896, 1024, dilation=16, dropout_rate=dropout_rate)

        # Decoder
        self.d1 = DecoderBlock(1024, 512)
        self.d2 = DecoderBlock(512, 256)
        self.d3 = DecoderBlock(256, 128)
        self.d4 = DecoderBlock(128, 64)

        # Define the final convolution to produce a single channel output
        self.final_conv = nn.Conv2d(64, 2, kernel_size=1, stride=1)

    def forward(self, inputs):
        # Encoder
        s1, p1 = self.e1(inputs)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)

        # Bottleneck
        b = self.b1(p4)
        b = self.b2(b)
        b = self.b3(b)
        b = self.b4(b)

        # Decoder
        d1 = self.d1(b, s4)
        d2 = self.d2(d1, s3)
        d3 = self.d3(d2, s2)
        d4 = self.d4(d3, s1)

        x = self.final_conv(d4)

        return x


class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(nn.Conv2d(in_planes, in_planes // 16, 1, bias=False),
                                nn.ReLU(),
                                nn.Conv2d(in_planes // 16, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)


class conv_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
        self.in1 = nn.InstanceNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.in2 = nn.InstanceNorm2d(out_c)

        self.relu = nn.ReLU()

        self.ca = ChannelAttention(out_c)
        self.sa = SpatialAttention()

    def forward(self, inputs):
        x = self.conv1(inputs)
        x = self.in1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.in2(x)
        x = self.relu(x)

        x = self.ca(x) * x
        x = self.sa(x) * x

        return x


class encoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv = conv_block(in_c, out_c)
        self.pool = nn.MaxPool2d((2, 2))

    def forward(self, inputs):
        x = self.conv(inputs)
        p = self.pool(x)

        return x, p


class decoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2, padding=0)
        self.conv = conv_block(out_c + out_c, out_c)

    def forward(self, inputs, skip):
        x = self.up(inputs)
        x = torch.cat([x, skip], axis=1)
        x = self.conv(x)
        return x


class build_unet(nn.Module):
    def __init__(self):
        super().__init__()

        """ Encoder """
        self.e1 = encoder_block(3, 64)
        self.e2 = encoder_block(64, 128)
        self.e3 = encoder_block(128, 256)
        self.e4 = encoder_block(256, 512)

        """ Bottleneck """
        self.b1 = conv_block(512, 640)
        self.b2 = conv_block(640, 768)
        self.b3 = conv_block(768, 896)
        self.b4 = conv_block(896, 1024)

        """ Decoder """
        self.d1 = decoder_block(1024, 512)
        self.d2 = decoder_block(512, 256)
        self.d3 = decoder_block(256, 128)
        self.d4 = decoder_block(128, 64)

        """ Classifier """
        self.outputs = outConv(64, 2)
        # self.outputs = nn.Conv2d(64, 1, kernel_size=1, padding=0)

    def forward(self, inputs):
        """ Encoder """
        s1, p1 = self.e1(inputs)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)

        """ Bottleneck """
        b = self.b1(p4)
        b = self.b2(b)
        b = self.b3(b)
        b = self.b4(b)

        """ Decoder """
        d1 = self.d1(b, s4)
        d2 = self.d2(d1, s3)
        d3 = self.d3(d2, s2)
        d4 = self.d4(d3, s1)

        outputs = self.outputs(d4)

        return outputs



class U_net(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_net, self).__init__()   # inherit nn.Module class
        self.classes = classes
        self.in_channels = in_channels

        # encoding
        self.Cov = DoubleConv(self.in_channels, 64)
        self.down1 = DownSample(64,128)
        self.down2 = DownSample(128, 256)
        self.down3 = DownSample(256, 512)
        self.down4 = DownSample(512, 1024)

        # decoding
        self.up1 = UpSample(1024,512)
        self.up2 = UpSample(512,256)
        self.up3 = UpSample(256,128)
        self.up4 = UpSample(128,64)

        # head
        self.out = outConv(64, self.classes)

    def forward(self, x):

        x1 = self.Cov(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x6 = self.up1(x4, x5)
        x7 = self.up2(x3, x6)
        x8 = self.up3(x2, x7)
        x9 = self.up4(x1, x8)

        x10 = self.out(x9)

        return x10


class unet_psp(nn.Module):
    def __init__(self, in_channels, classes):
        super(unet_psp, self).__init__()
        self.classes = classes
        self.in_channels = in_channels

        # encoding
        self.Cov = DoubleConv(self.in_channels, 64)

        self.psp1 = PSPool(64)
        self.Cov1 = DoubleConv(64, 128)

        self.down2 = DownSample(128, 256)
        self.down3 = DownSample(256, 512)
        self.down4 = DownSample(512, 1024)

        # decoding
        self.up1 = UpSample(1024,512)
        self.up2 = UpSample(512,256)
        self.up3 = UpSample(256,128)
        self.up4 = UpSample(128,64)

        # head
        self.out = outConv(64, self.classes)

    def forward(self, x):

        x1 = self.Cov(x)

        x2 = self.psp1(x1) # psp1
        x2 = self.Cov1(x2)

        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x6 = self.up1(x4, x5)
        x7 = self.up2(x3, x6)
        x8 = self.up3(x2, x7)
        x9 = self.up4(x1, x8)

        x10 = self.out(x9)
        return x10


class ResUNet(nn.Module):
    def __init__(self, in_channels, classes, dev):
        super(ResUNet, self).__init__()
        self.classes = classes
        self.in_channels = in_channels
        self.dev = dev

        self.up1 = UpSample(1024, 512)
        self.up2 = UpSample(512, 256)
        self.up3 = UpSample(256, 128)
        self.up4 = UpSample(128, 64)
        self.out = outConv(64, self.classes)

    def forward(self, x):

        md = ResNet50Encoder(in_channel=self.in_channels)
        md.to(device=self.dev)

        [x1, x2, x3, x4, x5] = md(x)

        x6 = self.up1(x4, x5)
        x7 = self.up2(x3, x6)
        x8 = self.up3(x2, x7)
        x9 = self.up4(x1, x8)
        x10 = self.out(x9)

        return x10



In [9]:
# Swin-Unet Architecture

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size[0], window_size[0], W // window_size[1], window_size[1], C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size[0], window_size[1], C)
    return windows

def window_reverse(windows, window_size, H, W):
    C = windows.shape[-1]
    x = windows.view(-1, H // window_size[0], W // window_size[1], window_size[0], window_size[1], C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, H, W, C)
    return x

def get_relative_position_index(win_h: int, win_w: int):
    # get pair-wise relative position index for each token inside the window
    coords = torch.stack(torch.meshgrid(torch.arange(win_h), torch.arange(win_w),indexing='ij'))  # 2, Wh, Ww
    coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
    relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
    relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
    relative_coords[:, :, 0] += win_h - 1  # shift to start from 0
    relative_coords[:, :, 1] += win_w - 1
    relative_coords[:, :, 0] *= 2 * win_w - 1
    return relative_coords.sum(-1)  # Wh*Ww, Wh*Ww

class WindowAttention(nn.Module):
    def __init__(
            self,
            dim,
            window_size,
    ):
        super().__init__()
        self.window_size = window_size
        self.window_area = self.window_size[0]*self.window_size[1]
        self.num_heads = 4
        head_dim =  dim // self.num_heads
        # attn_dim = head_dim * self.num_heads
        self.scale = head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(torch.zeros((2 * window_size[0] - 1) **2, self.num_heads))

        # get pair-wise relative position index for each token inside the window
        self.register_buffer("relative_position_index", get_relative_position_index(self.window_size[0], self.window_size[1]), persistent=False)

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        torch.nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def _get_rel_pos_bias(self):
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)].view(self.window_area, self.window_area, -1)  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wh*Ww, Wh*Ww
        return relative_position_bias.unsqueeze(0)

    def forward(self, x, mask = None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)


        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        attn = attn + self._get_rel_pos_bias()
        if mask is not None:
            num_win = mask.shape[0]
            attn = attn.view(-1, num_win, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.softmax(attn)
        x = attn @ v

        x = x.transpose(1, 2).reshape(B_, N, -1)
        x = self.proj(x)
        return x

class SwinTransformerBlock(nn.Module):
    def __init__(
            self,  dim, input_resolution, window_size = 7, shift_size = 0):

        super().__init__()
        self.input_resolution = input_resolution
        window_size = (window_size, window_size)
        shift_size = (shift_size, shift_size)
        self.window_size = window_size
        self.shift_size = shift_size
        self.window_area = self.window_size[0] * self.window_size[1]

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim,
            window_size=self.window_size,
        )

        self.norm2 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.LayerNorm(4 * dim),
            nn.Linear( 4 * dim, dim)
        )

        if self.shift_size:
            # calculate attention mask for SW-MSA
            H, W = self.input_resolution
            H = math.ceil(H / self.window_size[0]) * self.window_size[0]
            W = math.ceil(W / self.window_size[1]) * self.window_size[1]
            img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1
            cnt = 0
            for h in (
                    slice(0, -self.window_size[0]),
                    slice(-self.window_size[0], -self.shift_size[0]),
                    slice(-self.shift_size[0], None)):
                for w in (
                        slice(0, -self.window_size[1]),
                        slice(-self.window_size[1], -self.shift_size[1]),
                        slice(-self.shift_size[1], None)):
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            mask_windows = window_partition(img_mask, self.window_size)  # nW, window_size, window_size, 1
            mask_windows = mask_windows.view(-1, self.window_area)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask, persistent=False)

    def _attn(self, x):
        B, H, W, C = x.shape

        # cyclic shift
        if self.shift_size:
            shifted_x = torch.roll(x, shifts=(-self.shift_size[0], -self.shift_size[1]), dims=(1, 2))
        else:
            shifted_x = x

        # partition windows
        x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
        x_windows = x_windows.view(-1, self.window_area, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size[0], self.window_size[1], C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
        shifted_x = shifted_x[:, :H, :W, :].contiguous()

        # reverse cyclic shift
        if self.shift_size:
            x = torch.roll(shifted_x, shifts=self.shift_size, dims=(1, 2))
        else:
            x = shifted_x
        return x

    def forward(self, x):
        B, H, W, C = x.shape
        B, H, W, C = x.shape
        x = x + self._attn(self.norm1(x))
        x = x.reshape(B, -1, C)
        x = x + self.mlp(self.norm2(x))
        x = x.reshape(B, H,W, C)
        return x

class PatchEmbedding(nn.Module):
    def __init__(self, in_ch, num_feat, patch_size):
        super().__init__()
        self.conv = nn.Conv2d(in_ch,num_feat, kernel_size=patch_size,
                                  stride=patch_size)

    def forward(self, X):
        # Output shape: (batch size, no. of patches, no. of channels)
        return self.conv(X).permute(0,2,3,1)

class PatchMerging(nn.Module):

    def __init__(
            self,
            dim
    ):
        super().__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4*dim, 2*dim, bias=False)

    def forward(self, x):
        B, H, W, C = x.shape
        x = x.reshape(B, H // 2, 2, W // 2, 2, C).permute(0, 1, 3, 4, 2, 5).flatten(3)
        x = self.norm(x)
        x = self.reduction(x)
        return x

class PatchExpansion(nn.Module):

    def __init__(
            self,
            dim
    ):
        super().__init__()
        self.norm = nn.LayerNorm(dim//2)
        self.expand = nn.Linear(dim, 2*dim, bias=False)

    def forward(self, x):

        x = self.expand(x)
        B, H, W, C = x.shape

        x = x.view(B, H , W, 2, 2, C//4)
        x = x.permute(0,1,3,2,4,5)

        x = x.reshape(B,H*2, W*2 , C//4)

        x = self.norm(x)
        return x

class FinalPatchExpansion(nn.Module):

    def __init__(
            self,
            dim
    ):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.expand = nn.Linear(dim, 16*dim, bias=False)

    def forward(self, x):

        x = self.expand(x)
        B, H, W, C = x.shape

        x = x.view(B, H , W, 4, 4, C//16)
        x = x.permute(0,1,3,2,4,5)

        x = x.reshape(B,H*4, W*4 , C//16)

        x = self.norm(x)
        return x

class SwinBlock(nn.Module):
    def __init__(self, dims, ip_res, ss_size = 3):
        super().__init__()
        self.swtb1 = SwinTransformerBlock(dim=dims, input_resolution=ip_res)
        self.swtb2 = SwinTransformerBlock(dim=dims, input_resolution=ip_res, shift_size=ss_size)

    def forward(self, x):
        return self.swtb2(self.swtb1(x))


class Encoder(nn.Module):
    def __init__(self, C, partioned_ip_res, num_blocks=3):
        super().__init__()
        H,W = partioned_ip_res[0], partioned_ip_res[1]
        self.enc_swin_blocks = nn.ModuleList([
            SwinBlock(C, (H, W)),
            SwinBlock(2*C, (H//2, W//2)),
            SwinBlock(4*C, (H//4, W//4))
        ])
        self.enc_patch_merge_blocks = nn.ModuleList([
            PatchMerging(C),
            PatchMerging(2*C),
            PatchMerging(4*C)
        ])

    def forward(self, x):
        skip_conn_ftrs = []
        for swin_block,patch_merger in zip(self.enc_swin_blocks, self.enc_patch_merge_blocks):
            x = swin_block(x)
            skip_conn_ftrs.append(x)
            x = patch_merger(x)
        return x, skip_conn_ftrs


class Decoder(nn.Module):
    def __init__(self, C, partioned_ip_res, num_blocks=3):
        super().__init__()
        H,W = partioned_ip_res[0], partioned_ip_res[1]
        self.dec_swin_blocks = nn.ModuleList([
            SwinBlock(4*C, (H//4, W//4)),
            SwinBlock(2*C, (H//2, W//2)),
            SwinBlock(C, (H, W))
        ])
        self.dec_patch_expand_blocks = nn.ModuleList([
            PatchExpansion(8*C),
            PatchExpansion(4*C),
            PatchExpansion(2*C)
        ])
        self.skip_conn_concat = nn.ModuleList([
            nn.Linear(8*C, 4*C),
            nn.Linear(4*C, 2*C),
            nn.Linear(2*C, 1*C)
        ])

    def forward(self, x, encoder_features):
        for patch_expand,swin_block, enc_ftr, linear_concatter in zip(self.dec_patch_expand_blocks, self.dec_swin_blocks, encoder_features,self.skip_conn_concat):
            x = patch_expand(x)
            x = torch.cat([x, enc_ftr], dim=-1)
            x = linear_concatter(x)
            x = swin_block(x)
        return x


class SwinUNet(nn.Module):
    def __init__(self, H, W, ch, C, num_class, num_blocks=3, patch_size = 4):
        super().__init__()
        self.patch_embed = PatchEmbedding(ch, C, patch_size)
        self.encoder = Encoder(C, (H//patch_size, W//patch_size),num_blocks)
        self.bottleneck = SwinBlock(C*(2**num_blocks), (H//(patch_size* (2**num_blocks)), W//(patch_size* (2**num_blocks))))
        self.decoder = Decoder(C, (H//patch_size, W//patch_size),num_blocks)
        self.final_expansion = FinalPatchExpansion(C)
        self.head        = nn.Conv2d(C, num_class, 1,padding='same')

    def forward(self, x):
        x = self.patch_embed(x)

        x,skip_ftrs  = self.encoder(x)

        x = self.bottleneck(x)

        x = self.decoder(x, skip_ftrs[::-1])

        x = self.final_expansion(x)

        x = self.head(x.permute(0,3,1,2))

        return x

In [10]:

class HUMG_NEW(Dataset):
    def __init__(self, root_path, set_type='train', transform=None):
        assert set_type in ['train', 'val', 'test'], "set_type must be 'train', 'val', or 'test'"

        self.root_path = root_path
        self.set_type = set_type
        self.img_path = os.path.join(root_path, set_type, 'images')
        self.mask_path = os.path.join(root_path, set_type, 'labels')

        self.img_list = sorted([f for f in os.listdir(self.img_path) if f.endswith(('.png', '.jpg', '.tif'))])
        self.mask_list = sorted([f for f in os.listdir(self.mask_path) if f.endswith(('.png', '.jpg', '.tif'))])

        # Check for valid and matching files
        assert len(self.img_list) == len(self.mask_list), "Mismatch between images and masks"
        for img, mask in zip(self.img_list, self.mask_list):
            assert os.path.splitext(img)[0] == os.path.splitext(mask)[0], "Image and mask filenames do not match"

        assert len(self.img_list) > 0, f"No images found in {self.img_path}"
        assert len(self.mask_list) > 0, f"No masks found in {self.mask_path}"

        _trans_forms = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.transform = transform if transform else _trans_forms
        # self._default_transforms(set_type)

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, index):
        img_path = os.path.join(self.img_path, self.img_list[index])
        mask_path = os.path.join(self.mask_path, self.mask_list[index])

        img = Image.open(img_path).convert('RGB')  # Raw image for visualization
        mask = Image.open(mask_path).convert('L')  # Mask for training

        mask_array = np.array(mask, dtype=np.uint8)
        mask_binary = (mask_array == 255).astype(np.uint8)

        # Apply augmentations only during training
        if self.set_type == 'train':
            img, mask_binary = self._apply_augmentations(img, mask_binary)

        img_raw = np.array(img.copy(), dtype=np.uint8)

        # Apply transforms (e.g., normalization) only to the image
        img_tensor = self.transform(img)
        mask_tensor = torch.as_tensor(mask_binary, dtype=torch.long)

        # Return only the tensors for batching, not the raw image
        return {'img': img_tensor, 'mask': mask_tensor, 'raw_img': img_raw}  # 'raw_img' will be accessed later for visualization

    def _apply_augmentations(self, img, mask):
        target_size = (512, 512)
        img = transforms.functional.resize(img, target_size)
        mask = np.array(Image.fromarray(mask).resize(target_size, resample=Image.NEAREST))

        if random.random() < 0.5:
            img = transforms.functional.hflip(img)
            mask = np.fliplr(mask)
        if random.random() < 0.5:
            img = transforms.functional.vflip(img)
            mask = np.flipud(mask)

        color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)
        img = color_jitter(img)

        angle = random.uniform(-15, 15)
        img = transforms.functional.rotate(img, angle)
        mask = np.array(Image.fromarray(mask).rotate(angle, resample=Image.NEAREST))

        return img, mask

    def _default_transforms(self, set_type):
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

In [11]:

# Define encoder and weights
ENCODER = 'se_resnext50_32x4d'
ENCODER_WEIGHTS = 'imagenet'



# # Step 1: Define your custom encoder (e.g., a simple CNN)
# class CustomEncoder(nn.Module):
#     def __init__(self):
#         super(CustomEncoder, self).__init__()
#         self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1)
#         self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
#         self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)

#     def forward(self, x):
#         x = self.conv1(x)
#         x = self.conv2(x)
#         x = self.conv3(x)
#         return x

# # Step 2: Initialize the model with your custom encoder
# model = smp.Unet(
#     encoder_name=CustomEncoder,  # Pass your custom encoder class
#     encoder_weights=None,  # Random initialization of weights
#     classes=1,  # Number of output classes
#     activation='sigmoid'  # Use sigmoid for binary segmentation
# )

# Create segmentation model using SMP with a pretrained encoder
class U_rnet(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_rnet, self).__init__()  # inherit nn.Module class

        # SMP Unet with se_resnext50_32x4d as the encoder
        self.model = smp.Unet(
            encoder_name=ENCODER,                # Encoder: se_resnext50_32x4d
            encoder_weights=ENCODER_WEIGHTS,     # Pretrained on ImageNet
            in_channels=in_channels,             # Number of input channels
            classes=classes                      # Number of output classes
        )

    def forward(self, x):
        return self.model(x)

ENCODER = 'swin_transformer'  # Using the Swin Transformer encoder
ENCODER_WEIGHTS = 'imagenet'  # Pre-trained weights on ImageNet

# Create segmentation model using SMP with a pretrained encoder
class U_stnet(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_stnet, self).__init__()  # inherit nn.Module class

        # SMP Unet with se_resnext50_32x4d as the encoder
        self.model = smp.Unet(
            encoder_name=ENCODER,                # Encoder: se_resnext50_32x4d
            encoder_weights=ENCODER_WEIGHTS,     # Pretrained on ImageNet
            in_channels=in_channels,             # Number of input channels
            classes=classes                      # Number of output classes
        )

    def forward(self, x):
        return self.model(x)

# Define Encoder and Pretrained Weights
# ENCODER = 'swin_transformer'
ENCODER = 'timm-efficientnet-b7'  # Using EfficientNet-b7 encoder
ENCODER = 'se_resnext50_32x4d'
ENCODER_WEIGHTS = 'imagenet'

# Wrapper class for the segmentation model
class SegmentationModel(nn.Module):
    def __init__(self, architecture, encoder, in_channels, classes):
        super(SegmentationModel, self).__init__()

        # Choose the architecture dynamically
        if architecture == 'Unet':
            self.model = smp.Unet(
                encoder_name=encoder,
                encoder_weights=ENCODER_WEIGHTS,
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'DeepLabV3+':
            self.model = smp.DeepLabV3Plus(
                encoder_name=encoder,
                encoder_weights=ENCODER_WEIGHTS,
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'FPN':
            self.model = smp.FPN(
                encoder_name=encoder,
                encoder_weights=ENCODER_WEIGHTS,
                in_channels=in_channels,
                classes=classes
            )

        else:
            raise ValueError(f"Unknown architecture: {architecture}")

    def forward(self, x):
        return self.model(x)


# Wrapper class for the segmentation model
class SegmentationModel_NEW(nn.Module):
    # def __init__(self, architecture, in_channels, classes):
    def __init__(self, architecture, encoder_name, in_channels, classes):
        super(SegmentationModel_NEW, self).__init__()
        # Choose the architecture dynamically
        print (architecture)
        ENCODER_WEIGHTS = 'imagenet' # Kept imagenet as a default weight source
        if architecture == 'Unet':
            self.model = smp.Unet(
                encoder_name='resnet34', # Using a specific encoder name
                encoder_weights=None,    # Can be None or 'imagenet'
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'Unet++':
            self.model = smp.UnetPlusPlus(
                encoder_name='resnet34', # Using a specific encoder name
                encoder_weights=None, # Can be None or 'imagenet'
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'DeepLabV3':
            self.model = smp.DeepLabV3(
                encoder_name='resnet34', # Using a specific encoder name
                encoder_weights=None, # Can be None or 'imagenet'
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'DeepLabV3+':
            self.model = smp.DeepLabV3Plus(
                encoder_name='resnet34',  # Using a specific encoder name
                encoder_weights=None,     # Can be None or 'imagenet'
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'Segformer':
            # For Segformer, an encoder_name must be specified, None is not supported
            self.model = smp.Segformer(
                encoder_name=encoder_name, #'se-resnet50', # Using a valid Segformer encoder name
                encoder_weights=None,       # Can be None or a valid weight name
                in_channels=in_channels,
                classes=classes
            )
        elif architecture == 'FPN':
            self.model = smp.FPN(
                encoder_name='resnet34', # Using a specific encoder name
                encoder_weights=None, # Can be None or 'imagenet'
                in_channels=in_channels,
                classes=classes
            )
        else:
            raise ValueError(f"Unknown architecture: {architecture}")

    def forward(self, x):
        return self.model(x)

# Define the Residual Convolution Block with Atrous Convolutions
class ResConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation=1, dropout_rate=0.3):
        super(ResConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_rate)  # Dropout layer
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)  # Apply dropout
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        return out

# Define the CBAM Attention Blocks
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

# Define the Convolutional Block with Attention
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, dilation=1, dropout_rate=0.3):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_rate)  # Dropout layer
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.ca = ChannelAttention(out_c)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

# Define the Encoder and Decoder Blocks
class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c, dilation=1):
        super(EncoderBlock, self).__init__()
        self.conv = ConvBlock(in_c, out_c, dilation)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.conv(x)
        p = self.pool(x)
        return x, p

class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super(DecoderBlock, self).__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_c * 2, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        x = self._pad_to_match(x, skip)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

    def _pad_to_match(self, x, skip):
        """Pad x to match the spatial dimensions of skip."""
        if x.size(2) < skip.size(2):
            pad_h = skip.size(2) - x.size(2)
            pad_w = skip.size(3) - x.size(3)
            x = F.pad(x, (0, pad_w, 0, pad_h))
        return x

# Define the ResCBAM_UNet Model
class ResCBAM_UNet_v4(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(ResCBAM_UNet_v4, self).__init__()

        # Encoder
        self.e1 = EncoderBlock(3, 64)  # 3 channels for RGB input
        self.e2 = EncoderBlock(64, 128)
        self.e3 = EncoderBlock(128, 256)
        self.e4 = EncoderBlock(256, 512)

        # Bottleneck with Atrous Convolutions
        self.b1 = ResConvBlock(512, 640, dilation=2, dropout_rate=dropout_rate)
        self.b2 = ResConvBlock(640, 768, dilation=4, dropout_rate=dropout_rate)
        self.b3 = ResConvBlock(768, 896, dilation=8, dropout_rate=dropout_rate)
        self.b4 = ResConvBlock(896, 1024, dilation=16, dropout_rate=dropout_rate)

        # Decoder
        self.d1 = DecoderBlock(1024, 512)
        self.d2 = DecoderBlock(512, 256)
        self.d3 = DecoderBlock(256, 128)
        self.d4 = DecoderBlock(128, 64)

        # Define the final convolution to produce a single channel output
        self.final_conv = nn.Conv2d(64, 2, kernel_size=1, stride=1)

    def forward(self, inputs):
        # Encoder
        s1, p1 = self.e1(inputs)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)

        # Bottleneck
        b = self.b1(p4)
        b = self.b2(b)
        b = self.b3(b)
        b = self.b4(b)

        # Decoder
        d1 = self.d1(b, s4)
        d2 = self.d2(d1, s3)
        d3 = self.d3(d2, s2)
        d4 = self.d4(d3, s1)

        x = self.final_conv(d4)

        return x


class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(nn.Conv2d(in_planes, in_planes // 16, 1, bias=False),
                                nn.ReLU(),
                                nn.Conv2d(in_planes // 16, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)


class conv_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
        self.in1 = nn.InstanceNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.in2 = nn.InstanceNorm2d(out_c)

        self.relu = nn.ReLU()

        self.ca = ChannelAttention(out_c)
        self.sa = SpatialAttention()

    def forward(self, inputs):
        x = self.conv1(inputs)
        x = self.in1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.in2(x)
        x = self.relu(x)

        x = self.ca(x) * x
        x = self.sa(x) * x

        return x


class encoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv = conv_block(in_c, out_c)
        self.pool = nn.MaxPool2d((2, 2))

    def forward(self, inputs):
        x = self.conv(inputs)
        p = self.pool(x)

        return x, p


class decoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2, padding=0)
        self.conv = conv_block(out_c + out_c, out_c)

    def forward(self, inputs, skip):
        x = self.up(inputs)
        x = torch.cat([x, skip], axis=1)
        x = self.conv(x)
        return x


class build_unet(nn.Module):
    def __init__(self):
        super().__init__()

        """ Encoder """
        self.e1 = encoder_block(3, 64)
        self.e2 = encoder_block(64, 128)
        self.e3 = encoder_block(128, 256)
        self.e4 = encoder_block(256, 512)

        """ Bottleneck """
        self.b1 = conv_block(512, 640)
        self.b2 = conv_block(640, 768)
        self.b3 = conv_block(768, 896)
        self.b4 = conv_block(896, 1024)

        """ Decoder """
        self.d1 = decoder_block(1024, 512)
        self.d2 = decoder_block(512, 256)
        self.d3 = decoder_block(256, 128)
        self.d4 = decoder_block(128, 64)

        """ Classifier """
        self.outputs = outConv(64, 2)
        # self.outputs = nn.Conv2d(64, 1, kernel_size=1, padding=0)

    def forward(self, inputs):
        """ Encoder """
        s1, p1 = self.e1(inputs)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)

        """ Bottleneck """
        b = self.b1(p4)
        b = self.b2(b)
        b = self.b3(b)
        b = self.b4(b)

        """ Decoder """
        d1 = self.d1(b, s4)
        d2 = self.d2(d1, s3)
        d3 = self.d3(d2, s2)
        d4 = self.d4(d3, s1)

        outputs = self.outputs(d4)

        return outputs



class U_net(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_net, self).__init__()   # inherit nn.Module class
        self.classes = classes
        self.in_channels = in_channels

        # encoding
        self.Cov = DoubleConv(self.in_channels, 64)
        self.down1 = DownSample(64,128)
        self.down2 = DownSample(128, 256)
        self.down3 = DownSample(256, 512)
        self.down4 = DownSample(512, 1024)

        # decoding
        self.up1 = UpSample(1024,512)
        self.up2 = UpSample(512,256)
        self.up3 = UpSample(256,128)
        self.up4 = UpSample(128,64)

        # head
        self.out = outConv(64, self.classes)

    def forward(self, x):

        x1 = self.Cov(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x6 = self.up1(x4, x5)
        x7 = self.up2(x3, x6)
        x8 = self.up3(x2, x7)
        x9 = self.up4(x1, x8)

        x10 = self.out(x9)

        return x10

# Now, the U-Net Encoder
class U_net_Encoder(nn.Module):
    def __init__(self, in_channels, classes):
        super(U_net_Encoder, self).__init__()
        self.classes = classes
        self.in_channels = in_channels

        # encoding
        self.Cov = DoubleConv(self.in_channels, 64)
        self.down1 = DownSample(64,128)
        self.down2 = DownSample(128, 256)
        self.down3 = DownSample(256, 512)
        self.down4 = DownSample(512, 1024)

    def forward(self, x):
        x1 = self.Cov(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Return the intermediate features to be used by the decoder
        return [x1, x2, x3, x4, x5]

def get_custom_encoder(in_channels=3, depth=5, **kwargs):
    # Create an instance of your custom encoder (U-Net)
    encoder = U_net_Encoder(in_channels=in_channels, classes=kwargs.get("classes", 1))

    # Return the custom encoder, similar to how `get_encoder` returns the encoder
    return encoder

# Wrapper to use as a custom encoder in segmentation_models.pytorch
import segmentation_models_pytorch as smp

def get_unet_with_custom_encoder(in_channels, classes):
    # encoder = U_net_Encoder(in_channels, classes)
    # Get the custom encoder
    encoder = get_custom_encoder(in_channels=in_channels, classes=classes)

    # Now use this custom encoder with Unet
    model = smp.Unet(
        encoder_weights=None,  # Random initialization
        classes=classes,
        # activation='sigmoid',
        encoder=encoder  # Pass the custom encoder directly
    )
    return model

class unet_psp(nn.Module):
    def __init__(self, in_channels, classes):
        super(unet_psp, self).__init__()
        self.classes = classes
        self.in_channels = in_channels

        # encoding
        self.Cov = DoubleConv(self.in_channels, 64)

        self.psp1 = PSPool(64)
        self.Cov1 = DoubleConv(64, 128)

        self.down2 = DownSample(128, 256)
        self.down3 = DownSample(256, 512)
        self.down4 = DownSample(512, 1024)

        # decoding
        self.up1 = UpSample(1024,512)
        self.up2 = UpSample(512,256)
        self.up3 = UpSample(256,128)
        self.up4 = UpSample(128,64)

        # head
        self.out = outConv(64, self.classes)

    def forward(self, x):

        x1 = self.Cov(x)

        x2 = self.psp1(x1) # psp1
        x2 = self.Cov1(x2)

        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x6 = self.up1(x4, x5)
        x7 = self.up2(x3, x6)
        x8 = self.up3(x2, x7)
        x9 = self.up4(x1, x8)

        x10 = self.out(x9)
        return x10


class ResUNet(nn.Module):
    def __init__(self, in_channels, classes, dev):
        super(ResUNet, self).__init__()
        self.classes = classes
        self.in_channels = in_channels
        self.dev = dev

        self.up1 = UpSample(1024, 512)
        self.up2 = UpSample(512, 256)
        self.up3 = UpSample(256, 128)
        self.up4 = UpSample(128, 64)
        self.out = outConv(64, self.classes)

    def forward(self, x):

        md = ResNet50Encoder(in_channel=self.in_channels)
        md.to(device=self.dev)

        [x1, x2, x3, x4, x5] = md(x)

        x6 = self.up1(x4, x5)
        x7 = self.up2(x3, x6)
        x8 = self.up3(x2, x7)
        x9 = self.up4(x1, x8)
        x10 = self.out(x9)

        return x10



In [12]:
# Swin-Unet Architecture

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size[0], window_size[0], W // window_size[1], window_size[1], C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size[0], window_size[1], C)
    return windows

def window_reverse(windows, window_size, H, W):
    C = windows.shape[-1]
    x = windows.view(-1, H // window_size[0], W // window_size[1], window_size[0], window_size[1], C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, H, W, C)
    return x

def get_relative_position_index(win_h: int, win_w: int):
    # get pair-wise relative position index for each token inside the window
    coords = torch.stack(torch.meshgrid(torch.arange(win_h), torch.arange(win_w),indexing='ij'))  # 2, Wh, Ww
    coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
    relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
    relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
    relative_coords[:, :, 0] += win_h - 1  # shift to start from 0
    relative_coords[:, :, 1] += win_w - 1
    relative_coords[:, :, 0] *= 2 * win_w - 1
    return relative_coords.sum(-1)  # Wh*Ww, Wh*Ww

class WindowAttention(nn.Module):
    def __init__(
            self,
            dim,
            window_size,
    ):
        super().__init__()
        self.window_size = window_size
        self.window_area = self.window_size[0]*self.window_size[1]
        self.num_heads = 4
        head_dim =  dim // self.num_heads
        # attn_dim = head_dim * self.num_heads
        self.scale = head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(torch.zeros((2 * window_size[0] - 1) **2, self.num_heads))

        # get pair-wise relative position index for each token inside the window
        self.register_buffer("relative_position_index", get_relative_position_index(self.window_size[0], self.window_size[1]), persistent=False)

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        torch.nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def _get_rel_pos_bias(self):
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)].view(self.window_area, self.window_area, -1)  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wh*Ww, Wh*Ww
        return relative_position_bias.unsqueeze(0)

    def forward(self, x, mask = None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)


        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        attn = attn + self._get_rel_pos_bias()
        if mask is not None:
            num_win = mask.shape[0]
            attn = attn.view(-1, num_win, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.softmax(attn)
        x = attn @ v

        x = x.transpose(1, 2).reshape(B_, N, -1)
        x = self.proj(x)
        return x

class SwinTransformerBlock(nn.Module):
    def __init__(
            self,  dim, input_resolution, window_size = 7, shift_size = 0):

        super().__init__()
        self.input_resolution = input_resolution
        window_size = (window_size, window_size)
        shift_size = (shift_size, shift_size)
        self.window_size = window_size
        self.shift_size = shift_size
        self.window_area = self.window_size[0] * self.window_size[1]

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim,
            window_size=self.window_size,
        )

        self.norm2 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.LayerNorm(4 * dim),
            nn.Linear( 4 * dim, dim)
        )

        if self.shift_size:
            # calculate attention mask for SW-MSA
            H, W = self.input_resolution
            H = math.ceil(H / self.window_size[0]) * self.window_size[0]
            W = math.ceil(W / self.window_size[1]) * self.window_size[1]
            img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1
            cnt = 0
            for h in (
                    slice(0, -self.window_size[0]),
                    slice(-self.window_size[0], -self.shift_size[0]),
                    slice(-self.shift_size[0], None)):
                for w in (
                        slice(0, -self.window_size[1]),
                        slice(-self.window_size[1], -self.shift_size[1]),
                        slice(-self.shift_size[1], None)):
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            mask_windows = window_partition(img_mask, self.window_size)  # nW, window_size, window_size, 1
            mask_windows = mask_windows.view(-1, self.window_area)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask, persistent=False)

    def _attn(self, x):
        B, H, W, C = x.shape

        # cyclic shift
        if self.shift_size:
            shifted_x = torch.roll(x, shifts=(-self.shift_size[0], -self.shift_size[1]), dims=(1, 2))
        else:
            shifted_x = x

        # partition windows
        x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
        x_windows = x_windows.view(-1, self.window_area, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size[0], self.window_size[1], C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
        shifted_x = shifted_x[:, :H, :W, :].contiguous()

        # reverse cyclic shift
        if self.shift_size:
            x = torch.roll(shifted_x, shifts=self.shift_size, dims=(1, 2))
        else:
            x = shifted_x
        return x

    def forward(self, x):
        B, H, W, C = x.shape
        B, H, W, C = x.shape
        x = x + self._attn(self.norm1(x))
        x = x.reshape(B, -1, C)
        x = x + self.mlp(self.norm2(x))
        x = x.reshape(B, H,W, C)
        return x

class PatchEmbedding(nn.Module):
    def __init__(self, in_ch, num_feat, patch_size):
        super().__init__()
        self.conv = nn.Conv2d(in_ch,num_feat, kernel_size=patch_size,
                                  stride=patch_size)

    def forward(self, X):
        # Output shape: (batch size, no. of patches, no. of channels)
        return self.conv(X).permute(0,2,3,1)

class PatchMerging(nn.Module):

    def __init__(
            self,
            dim
    ):
        super().__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4*dim, 2*dim, bias=False)

    def forward(self, x):
        B, H, W, C = x.shape
        x = x.reshape(B, H // 2, 2, W // 2, 2, C).permute(0, 1, 3, 4, 2, 5).flatten(3)
        x = self.norm(x)
        x = self.reduction(x)
        return x

class PatchExpansion(nn.Module):

    def __init__(
            self,
            dim
    ):
        super().__init__()
        self.norm = nn.LayerNorm(dim//2)
        self.expand = nn.Linear(dim, 2*dim, bias=False)

    def forward(self, x):

        x = self.expand(x)
        B, H, W, C = x.shape

        x = x.view(B, H , W, 2, 2, C//4)
        x = x.permute(0,1,3,2,4,5)

        x = x.reshape(B,H*2, W*2 , C//4)

        x = self.norm(x)
        return x

class FinalPatchExpansion(nn.Module):

    def __init__(
            self,
            dim
    ):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.expand = nn.Linear(dim, 16*dim, bias=False)

    def forward(self, x):

        x = self.expand(x)
        B, H, W, C = x.shape

        x = x.view(B, H , W, 4, 4, C//16)
        x = x.permute(0,1,3,2,4,5)

        x = x.reshape(B,H*4, W*4 , C//16)

        x = self.norm(x)
        return x

class SwinBlock(nn.Module):
    def __init__(self, dims, ip_res, ss_size = 3):
        super().__init__()
        self.swtb1 = SwinTransformerBlock(dim=dims, input_resolution=ip_res)
        self.swtb2 = SwinTransformerBlock(dim=dims, input_resolution=ip_res, shift_size=ss_size)

    def forward(self, x):
        return self.swtb2(self.swtb1(x))


class Encoder(nn.Module):
    def __init__(self, C, partioned_ip_res, num_blocks=3):
        super().__init__()
        H,W = partioned_ip_res[0], partioned_ip_res[1]
        self.enc_swin_blocks = nn.ModuleList([
            SwinBlock(C, (H, W)),
            SwinBlock(2*C, (H//2, W//2)),
            SwinBlock(4*C, (H//4, W//4))
        ])
        self.enc_patch_merge_blocks = nn.ModuleList([
            PatchMerging(C),
            PatchMerging(2*C),
            PatchMerging(4*C)
        ])

    def forward(self, x):
        skip_conn_ftrs = []
        for swin_block,patch_merger in zip(self.enc_swin_blocks, self.enc_patch_merge_blocks):
            x = swin_block(x)
            skip_conn_ftrs.append(x)
            x = patch_merger(x)
        return x, skip_conn_ftrs


class Decoder(nn.Module):
    def __init__(self, C, partioned_ip_res, num_blocks=3):
        super().__init__()
        H,W = partioned_ip_res[0], partioned_ip_res[1]
        self.dec_swin_blocks = nn.ModuleList([
            SwinBlock(4*C, (H//4, W//4)),
            SwinBlock(2*C, (H//2, W//2)),
            SwinBlock(C, (H, W))
        ])
        self.dec_patch_expand_blocks = nn.ModuleList([
            PatchExpansion(8*C),
            PatchExpansion(4*C),
            PatchExpansion(2*C)
        ])
        self.skip_conn_concat = nn.ModuleList([
            nn.Linear(8*C, 4*C),
            nn.Linear(4*C, 2*C),
            nn.Linear(2*C, 1*C)
        ])

    def forward(self, x, encoder_features):
        for patch_expand,swin_block, enc_ftr, linear_concatter in zip(self.dec_patch_expand_blocks, self.dec_swin_blocks, encoder_features,self.skip_conn_concat):
            x = patch_expand(x)
            x = torch.cat([x, enc_ftr], dim=-1)
            x = linear_concatter(x)
            x = swin_block(x)
        return x


class SwinUNet(nn.Module):
    def __init__(self, H, W, ch, C, num_class, num_blocks=3, patch_size = 4):
        super().__init__()
        self.patch_embed = PatchEmbedding(ch, C, patch_size)
        self.encoder = Encoder(C, (H//patch_size, W//patch_size),num_blocks)
        self.bottleneck = SwinBlock(C*(2**num_blocks), (H//(patch_size* (2**num_blocks)), W//(patch_size* (2**num_blocks))))
        self.decoder = Decoder(C, (H//patch_size, W//patch_size),num_blocks)
        self.final_expansion = FinalPatchExpansion(C)
        self.head        = nn.Conv2d(C, num_class, 1,padding='same')

    def forward(self, x):
        x = self.patch_embed(x)

        x,skip_ftrs  = self.encoder(x)

        x = self.bottleneck(x)

        x = self.decoder(x, skip_ftrs[::-1])

        x = self.final_expansion(x)

        x = self.head(x.permute(0,3,1,2))

        return x

# Tao Average Meter

In [13]:
class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count



# Lập hàm tính độ chính xác IoU and Dice

In [14]:
#metrics
def intersectionAndUnionGPU(output, target, K, ignore_index=255):
    # 'K' classes, output and target sizes are N or N * L or N * H * W, each value in range 0 to K - 1.
    assert (output.dim() in [1, 2, 3])
    assert output.shape == target.shape
    output = output.view(-1)
    target = target.view(-1)
    output[target == ignore_index] = ignore_index
    intersection = output[output == target]
    area_intersection = torch.histc(intersection, bins=K, min=0, max=K-1)
    area_output = torch.histc(output, bins=K, min=0, max=K-1)
    area_target = torch.histc(target, bins=K, min=0, max=K-1)
    area_union = area_output + area_target - area_intersection
    return area_intersection, area_union, area_target

## TN, TP, FN, FP

In [15]:
# Parameters
in_channels, classes = 3, 2

data_path = '/content/drive/MyDrive/B2024_MDA_09_Demo/Modern_buildings(48)' # train, val, test
model_path = '/content/drive/MyDrive/B2024_MDA_09_Demo/checkpoint' # checkpoint_AdamW_Cosine_celoss_100_0.001

# Prepare Dataloader
test_dataloader = DataLoader(HUMG_NEW(data_path, set_type="test"), batch_size=1, shuffle=False, drop_last=False, pin_memory=True)
print('Len of testdataset',len(test_dataloader))

# Set max_images to the length of the entire test dataset
max_images = len(test_dataloader)
dataloader_length = len(test_dataloader)

# Get the first batch to print lengths outside the loop
first_batch = next(iter(test_dataloader))
images, masks = first_batch['img'], first_batch['mask']
print(f'Length of images tensor: {len(images)}')
print(f'Length of masks tensor: {len(masks)}')

# Predict and save results
import random
import torchmetrics
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
import numpy as np
import pandas as pd
import torch
import os

random.seed(42)

model1=ResCBAM_UNet_v4()
model2 = get_unet_with_custom_encoder(in_channels, classes)
model3 = SegmentationModel_NEW('Segformer','resnet34', in_channels, classes)
model4 = SegmentationModel_NEW('FPN','resnet34',in_channels, classes)


ckpt_path1 = f'{model_path}/ResCBAM_UNet_v4_checkpoint_epoch_50_NO_laptop.pth'
ckpt_path2 = f'{model_path}/SegmentationModel_Unet_custom_checkpoint_epoch_50_NO_laptop.pth'
ckpt_path3 = f'{model_path}/SegmentationModel_SegFormer_best_model_laptop_NO.pth'
ckpt_path4 = f'{model_path}/SegmentationModel_FPN_best_model_laptop_NO.pth'

device = torch.device('cpu')

state_dict1 = torch.load(ckpt_path1, map_location=device)
model1.load_state_dict(state_dict1)
model1.to(device=device)
model1.eval()

state_dict2 = torch.load(ckpt_path2, map_location=device)
model2.load_state_dict(state_dict2)
model2.to(device=device)
model2.eval()

state_dict3 = torch.load(ckpt_path3, map_location=device)
model3.load_state_dict(state_dict3)
model3.to(device=device)
model3.eval()

state_dict4 = torch.load(ckpt_path4, map_location=device)
model4.load_state_dict(state_dict4)
model4.to(device=device)
model4.eval()


# Instantiate all necessary metric meters for each model
test_acc_meter1 = AverageMeter()
test_precision_meter1 = AverageMeter()
test_recall_meter1 = AverageMeter()
test_F1Score_meter1 = AverageMeter()
test_iou_meter1 = AverageMeter()
test_tn_meter1 = AverageMeter()
test_tp_meter1 = AverageMeter()
test_fp_meter1 = AverageMeter()
test_fn_meter1 = AverageMeter()

test_acc_meter2 = AverageMeter()
test_precision_meter2 = AverageMeter()
test_recall_meter2 = AverageMeter()
test_F1Score_meter2 = AverageMeter()
test_iou_meter2 = AverageMeter()
test_tn_meter2 = AverageMeter()
test_tp_meter2 = AverageMeter()
test_fp_meter2 = AverageMeter()
test_fn_meter2 = AverageMeter()

test_acc_meter3 = AverageMeter()
test_precision_meter3 = AverageMeter()
test_recall_meter3 = AverageMeter()
test_F1Score_meter3 = AverageMeter()
test_iou_meter3 = AverageMeter()
test_tn_meter3 = AverageMeter()
test_tp_meter3 = AverageMeter()
test_fp_meter3 = AverageMeter()
test_fn_meter3 = AverageMeter()

test_acc_meter4 = AverageMeter()
test_precision_meter4 = AverageMeter()
test_recall_meter4 = AverageMeter()
test_F1Score_meter4 = AverageMeter()
test_iou_meter4 = AverageMeter()
test_tn_meter4 = AverageMeter()
test_tp_meter4 = AverageMeter()
test_fp_meter4 = AverageMeter()
test_fn_meter4 = AverageMeter()

predictions1 = []
predictions2 = []
predictions3 = []
predictions4 = []

# Instantiate metric functions
accuracy_fn = torchmetrics.Accuracy(task='multiclass', num_classes=classes, average='macro').to(device)
precision_fn = torchmetrics.Precision(task='multiclass', num_classes=classes, average='macro').to(device)
recall_fn = torchmetrics.Recall(task='multiclass', num_classes=classes, average='macro').to(device)
f1_fn = torchmetrics.F1Score(task='multiclass', num_classes=classes, average='macro').to(device)
iou_fn = torchmetrics.JaccardIndex(task='multiclass', num_classes=classes, average='macro').to(device)
# Metric function for confusion matrix components
conf_matrix_fn = torchmetrics.ConfusionMatrix(task='multiclass', num_classes=classes).to(device)


# Initialize a list to hold data for plotting
iou_scores_to_plot = []
confusion_maps_to_plot = []
confusion_metrics_to_plot = []

# Generate the color-coded confusion maps
def create_confusion_map(pred_mask, gt_mask):
   # Convert to numpy arrays for easier manipulation
   pred_mask_np = pred_mask.squeeze().cpu().numpy()
   gt_mask_np = gt_mask.squeeze().cpu().numpy()

   # Create a 3-channel (RGB) image initialized to black
   confusion_map = np.zeros((*pred_mask_np.shape, 3), dtype=np.uint8)

   # True Positives (TP): correctly identified buildings (pred=1, gt=1)
   tp_mask = (pred_mask_np == 1) & (gt_mask_np == 1)
   confusion_map[tp_mask] = [0, 0, 255] # Blue

   # True Negatives (TN): correctly identified background (pred=0, gt=0)
   tn_mask = (pred_mask_np == 0) & (gt_mask_np == 0)
   confusion_map[tn_mask] = [70, 70, 70] # Dark Gray

   # False Positives (FP): background incorrectly identified as a building (pred=1, gt=0)
   fp_mask = (pred_mask_np == 1) & (gt_mask_np == 0)
   confusion_map[fp_mask] = [255, 0, 0] # Red

   # False Negatives (FN): buildings incorrectly identified as background (pred=0, gt=1)
   fn_mask = (pred_mask_np == 0) & (gt_mask_np == 1)
   confusion_map[fn_mask] = [255, 255, 0] # Yellow

   return Image.fromarray(confusion_map)


for idx, batch in enumerate(tqdm(test_dataloader, unit='batch', total=max_images)):
   if idx >= max_images:
       break

   images, masks = batch['img'], batch['mask']
   images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
   raw_images = batch['raw_img']
   masks = masks.to(device=device, dtype=torch.long)

   with torch.no_grad():
       # Predict masks
       pred_mask_tensor1 = model1(images)
       pred_mask1_tensor = torch.argmax(pred_mask_tensor1, dim=1)

       pred_mask_tensor2 = model2(images)
       pred_mask2_tensor = torch.argmax(pred_mask_tensor2, dim=1)

       pred_mask_tensor3 = model3(images)
       pred_mask3_tensor = torch.argmax(pred_mask_tensor3, dim=1)

       pred_mask_tensor4 = model4(images)
       pred_mask4_tensor = torch.argmax(pred_mask_tensor4, dim=1)

       # Update metrics for Model 1
       conf_matrix1 = conf_matrix_fn(pred_mask1_tensor, masks).cpu().numpy()
       tn1, fp1, fn1, tp1 = conf_matrix1.ravel()
       test_acc_meter1.update(accuracy_fn(pred_mask1_tensor, masks).item(), masks.size(0))
       test_precision_meter1.update(precision_fn(pred_mask1_tensor, masks).item(), masks.size(0))
       test_recall_meter1.update(recall_fn(pred_mask1_tensor, masks).item(), masks.size(0))
       test_F1Score_meter1.update(f1_fn(pred_mask1_tensor, masks).item(), masks.size(0))
       test_iou_meter1.update(iou_fn(pred_mask1_tensor, masks).item(), masks.size(0))
       test_tn_meter1.update(tn1, 1)
       test_tp_meter1.update(tp1, 1)
       test_fp_meter1.update(fp1, 1)
       test_fn_meter1.update(fn1, 1)


       # Update metrics for Model 2
       conf_matrix2 = conf_matrix_fn(pred_mask2_tensor, masks).cpu().numpy()
       tn2, fp2, fn2, tp2 = conf_matrix2.ravel()
       test_acc_meter2.update(accuracy_fn(pred_mask2_tensor, masks).item(), masks.size(0))
       test_precision_meter2.update(precision_fn(pred_mask2_tensor, masks).item(), masks.size(0))
       test_recall_meter2.update(recall_fn(pred_mask2_tensor, masks).item(), masks.size(0))
       test_F1Score_meter2.update(f1_fn(pred_mask2_tensor, masks).item(), masks.size(0))
       test_iou_meter2.update(iou_fn(pred_mask2_tensor, masks).item(), masks.size(0))
       test_tn_meter2.update(tn2, 1)
       test_tp_meter2.update(tp2, 1)
       test_fp_meter2.update(fp2, 1)
       test_fn_meter2.update(fn2, 1)


       # Update metrics for Model 3
       conf_matrix3 = conf_matrix_fn(pred_mask3_tensor, masks).cpu().numpy()
       tn3, fp3, fn3, tp3 = conf_matrix3.ravel()
       test_acc_meter3.update(accuracy_fn(pred_mask3_tensor, masks).item(), masks.size(0))
       test_precision_meter3.update(precision_fn(pred_mask3_tensor, masks).item(), masks.size(0))
       test_recall_meter3.update(recall_fn(pred_mask3_tensor, masks).item(), masks.size(0))
       test_F1Score_meter3.update(f1_fn(pred_mask3_tensor, masks).item(), masks.size(0))
       test_iou_meter3.update(iou_fn(pred_mask3_tensor, masks).item(), masks.size(0))
       test_tn_meter3.update(tn3, 1)
       test_tp_meter3.update(tp3, 1)
       test_fp_meter3.update(fp3, 1)
       test_fn_meter3.update(fn3, 1)


       # Update metrics for Model 4
       conf_matrix4 = conf_matrix_fn(pred_mask4_tensor, masks).cpu().numpy()
       tn4, fp4, fn4, tp4 = conf_matrix4.ravel()
       test_acc_meter4.update(accuracy_fn(pred_mask4_tensor, masks).item(), masks.size(0))
       test_precision_meter4.update(precision_fn(pred_mask4_tensor, masks).item(), masks.size(0))
       test_recall_meter4.update(recall_fn(pred_mask4_tensor, masks).item(), masks.size(0))
       test_F1Score_meter4.update(f1_fn(pred_mask4_tensor, masks).item(), masks.size(0))
       test_iou_meter4.update(iou_fn(pred_mask4_tensor, masks).item(), masks.size(0))
       test_tn_meter4.update(tn4, 1)
       test_tp_meter4.update(tp4, 1)
       test_fp_meter4.update(fp4, 1)
       test_fn_meter4.update(fn4, 1)

   # Save results for visualization
   for i in range(images.shape[0]):
       ground_truth_image_pil = raw_images[i]
       ground_truth_mask_pil = Image.fromarray(masks[i].squeeze().cpu().numpy().astype(np.uint8))
       predicted_mask1_pil = Image.fromarray(pred_mask1_tensor[i].squeeze().cpu().numpy().astype(np.uint8))
       predicted_mask2_pil = Image.fromarray(pred_mask2_tensor[i].squeeze().cpu().numpy().astype(np.uint8))
       predicted_mask3_pil = Image.fromarray(pred_mask3_tensor[i].squeeze().cpu().numpy().astype(np.uint8))
       predicted_mask4_pil = Image.fromarray(pred_mask4_tensor[i].squeeze().cpu().numpy().astype(np.uint8))

       # Calculate individual IoU scores for plotting
       iou_1 = iou_fn(pred_mask1_tensor[i].unsqueeze(0), masks[i].unsqueeze(0)).item()
       iou_2 = iou_fn(pred_mask2_tensor[i].unsqueeze(0), masks[i].unsqueeze(0)).item()
       iou_3 = iou_fn(pred_mask3_tensor[i].unsqueeze(0), masks[i].unsqueeze(0)).item()
       iou_4 = iou_fn(pred_mask4_tensor[i].unsqueeze(0), masks[i].unsqueeze(0)).item()

       iou_scores_to_plot.append({
           'image': ground_truth_image_pil,
           'gt_mask': ground_truth_mask_pil,
           'pred_mask1': predicted_mask1_pil,
           'pred_mask2': predicted_mask2_pil,
           'pred_mask3': predicted_mask3_pil,
           'pred_mask4': predicted_mask4_pil,
           'iou_1': iou_1,
           'iou_2': iou_2,
           'iou_3': iou_3,
           'iou_4': iou_4,
       })

       # Generate confusion maps for each model
       def create_confusion_map(pred_mask, gt_mask):
           # Convert to numpy arrays for easier manipulation
           pred_mask_np = pred_mask.squeeze().cpu().numpy()
           gt_mask_np = gt_mask.squeeze().cpu().numpy()

           # Create a 3-channel (RGB) image initialized to black
           confusion_map = np.zeros((*pred_mask_np.shape, 3), dtype=np.uint8)

           # True Positives (TP): correctly identified buildings (pred=1, gt=1)
           tp_mask = (pred_mask_np == 1) & (gt_mask_np == 1)
           confusion_map[tp_mask] = [0, 0, 255] # Blue

           # True Negatives (TN): correctly identified background (pred=0, gt=0)
           tn_mask = (pred_mask_np == 0) & (gt_mask_np == 0)
           confusion_map[tn_mask] = [70, 70, 70] # Dark Gray

           # False Positives (FP): background incorrectly identified as a building (pred=1, gt=0)
           fp_mask = (pred_mask_np == 1) & (gt_mask_np == 0)
           confusion_map[fp_mask] = [255, 0, 0] # Red

           # False Negatives (FN): buildings incorrectly identified as background (pred=0, gt=1)
           fn_mask = (pred_mask_np == 0) & (gt_mask_np == 1)
           confusion_map[fn_mask] = [255, 255, 0] # Yellow

           return Image.fromarray(confusion_map)

       confusion_map1_pil = create_confusion_map(pred_mask1_tensor[i], masks[i])
       confusion_map2_pil = create_confusion_map(pred_mask2_tensor[i], masks[i])
       confusion_map3_pil = create_confusion_map(pred_mask3_tensor[i], masks[i])
       confusion_map4_pil = create_confusion_map(pred_mask4_tensor[i], masks[i])

       confusion_maps_to_plot.append({
           'image': ground_truth_image_pil,
           'gt_mask': ground_truth_mask_pil,
           'conf_map1': confusion_map1_pil,
           'conf_map2': confusion_map2_pil,
           'conf_map3': confusion_map3_pil,
           'conf_map4': confusion_map4_pil,
       })

# Print average IoU for the whole test set
print('TEST: IoU_1={:.3f}, IoU_2={:.3f}, IoU_3={:.3f}, IoU_4={:.3f}'.format(
   test_iou_meter1.avg,test_iou_meter2.avg,test_iou_meter3.avg,test_iou_meter4.avg))


# Final print statements for all models
print('--- Astro-ResCBAM-UNet ---')
print('TEST: Acc={:.3f}, Precision={:.3f}, Recall={:.3f}, F1_score={:.3f}, IoU={:.3f}'.format(
   test_acc_meter1.avg, test_precision_meter1.avg, test_recall_meter1.avg,
   test_F1Score_meter1.avg, test_iou_meter1.avg
))
print('--- Unet ---')
print('TEST: Acc={:.3f}, Precision={:.3f}, Recall={:.3f}, F1_score={:.3f}, IoU={:.3f}'.format(
   test_acc_meter2.avg, test_precision_meter2.avg, test_recall_meter2.avg,
   test_F1Score_meter2.avg, test_iou_meter2.avg
))
print('--- Segformer ---')
print('TEST: Acc={:.3f}, Precision={:.3f}, Recall={:.3f}, F1_score={:.3f}, IoU={:.3f}'.format(
   test_acc_meter3.avg, test_precision_meter3.avg, test_recall_meter3.avg,
   test_F1Score_meter3.avg, test_iou_meter3.avg
))
print('--- FPN ---')
print('TEST: Acc={:.3f}, Precision={:.3f}, Recall={:.3f}, F1_score={:.3f}, IoU={:.3f}'.format(
   test_acc_meter4.avg, test_precision_meter4.avg, test_recall_meter4.avg,
   test_F1Score_meter4.avg, test_iou_meter4.avg
))


# Define the output path for all saved files
output_path = os.path.join(data_path, 'test')
if not os.path.exists(output_path):
   os.makedirs(output_path)

# Define a batch size for plotting
plot_batch_size = 5
num_predictions_to_plot = len(iou_scores_to_plot)

# Loop through and plot the images in batches
for i in range(0, num_predictions_to_plot, plot_batch_size):
   start_index = i
   end_index = min(i + plot_batch_size, num_predictions_to_plot)
   current_batch_size = end_index - start_index

   # Create a new figure for each batch
   fig, axes = plt.subplots(current_batch_size * 2, 6, figsize=(18, 8 * current_batch_size))
   plt.style.use('seaborn-v0_8-white')

   if current_batch_size == 1:
       axes = axes.reshape(2, -1)

   model_names = ['Astro-ResCBAM-UNet', 'Unet', 'Segformer', 'FPN']

   # Loop through the images in the current batch
   for j in range(current_batch_size):
       data_idx = start_index + j
       data_iou = iou_scores_to_plot[data_idx]
       data_cm = confusion_maps_to_plot[data_idx]

       row_idx_pred = j * 2
       row_idx_cm = j * 2 + 1

       # Plotting logic for predicted masks
       axes[row_idx_pred, 0].set_title('Original Image', fontsize=12, pad=10)
       axes[row_idx_pred, 1].set_title('Ground Truth', fontsize=12, pad=10)
       axes[row_idx_pred, 2].set_title(model_names[0], fontsize=12, pad=10)
       axes[row_idx_pred, 3].set_title(model_names[1], fontsize=12, pad=10)
       axes[row_idx_pred, 4].set_title(model_names[2], fontsize=12, pad=10)
       axes[row_idx_pred, 5].set_title(model_names[3], fontsize=12, pad=10)

       axes[row_idx_pred, 0].imshow(data_iou['image'])
       axes[row_idx_pred, 0].axis('off')
       axes[row_idx_pred, 1].imshow(data_iou['gt_mask'], cmap='gray')
       axes[row_idx_pred, 1].axis('off')
       axes[row_idx_pred, 2].imshow(data_iou['pred_mask1'], cmap='gray')
       axes[row_idx_pred, 2].axis('off')
       axes[row_idx_pred, 2].text(0.5, -0.1, f"IoU: {data_iou['iou_1']:.3f}", size=12, ha="center", transform=axes[row_idx_pred, 2].transAxes)
       axes[row_idx_pred, 3].imshow(data_iou['pred_mask2'], cmap='gray')
       axes[row_idx_pred, 3].axis('off')
       axes[row_idx_pred, 3].text(0.5, -0.1, f"IoU: {data_iou['iou_2']:.3f}", size=12, ha="center", transform=axes[row_idx_pred, 3].transAxes)
       axes[row_idx_pred, 4].imshow(data_iou['pred_mask3'], cmap='gray')
       axes[row_idx_pred, 4].axis('off')
       axes[row_idx_pred, 4].text(0.5, -0.1, f"IoU: {data_iou['iou_3']:.3f}", size=12, ha="center", transform=axes[row_idx_pred, 4].transAxes)
       axes[row_idx_pred, 5].imshow(data_iou['pred_mask4'], cmap='gray')
       axes[row_idx_pred, 5].axis('off')
       axes[row_idx_pred, 5].text(0.5, -0.1, f"IoU: {data_iou['iou_4']:.3f}", size=12, ha="center", transform=axes[row_idx_pred, 5].transAxes)

       # Plotting logic for confusion maps
       axes[row_idx_cm, 0].set_title('Original Image', fontsize=12, pad=10)
       axes[row_idx_cm, 1].set_title('Ground Truth', fontsize=12, pad=10)
       axes[row_idx_cm, 2].set_title(model_names[0], fontsize=12, pad=10)
       axes[row_idx_cm, 3].set_title(model_names[1], fontsize=12, pad=10)
       axes[row_idx_cm, 4].set_title(model_names[2], fontsize=12, pad=10)
       axes[row_idx_cm, 5].set_title(model_names[3], fontsize=12, pad=10)

       axes[row_idx_cm, 0].imshow(data_cm['image'])
       axes[row_idx_cm, 0].axis('off')
       axes[row_idx_cm, 1].imshow(data_cm['gt_mask'], cmap='gray')
       axes[row_idx_cm, 1].axis('off')
       axes[row_idx_cm, 2].imshow(data_cm['conf_map1'])
       axes[row_idx_cm, 2].axis('off')
       axes[row_idx_cm, 2].text(0.5, -0.1, f"IoU: {data_iou['iou_1']:.3f}", size=12, ha="center", transform=axes[row_idx_cm, 2].transAxes)
       axes[row_idx_cm, 3].imshow(data_cm['conf_map2'])
       axes[row_idx_cm, 3].axis('off')
       axes[row_idx_cm, 3].text(0.5, -0.1, f"IoU: {data_iou['iou_2']:.3f}", size=12, ha="center", transform=axes[row_idx_cm, 3].transAxes)
       axes[row_idx_cm, 4].imshow(data_cm['conf_map3'])
       axes[row_idx_cm, 4].axis('off')
       axes[row_idx_cm, 4].text(0.5, -0.1, f"IoU: {data_iou['iou_3']:.3f}", size=12, ha="center", transform=axes[row_idx_cm, 4].transAxes)
       axes[row_idx_cm, 5].imshow(data_cm['conf_map4'])
       axes[row_idx_cm, 5].axis('off')
       axes[row_idx_cm, 5].text(0.5, -0.1, f"IoU: {data_iou['iou_4']:.3f}", size=12, ha="center", transform=axes[row_idx_cm, 5].transAxes)

   legend_elements = [
       Patch(facecolor='blue', edgecolor='k', label='True Positive (TP)'),
       Patch(facecolor='darkgray', edgecolor='k', label='True Negative (TN)'),
       Patch(facecolor='red', edgecolor='k', label='False Positive (FP)'),
       Patch(facecolor='yellow', edgecolor='k', label='False Negative (FN)'),
   ]
   fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.05), ncol=4)

   plt.tight_layout(pad=3.0, rect=[0, 0.05, 1, 1])

   # Save the file with a unique name
  #  plot_filename = f'segmentation_results_part_{i//plot_batch_size}.png'
  #  plt.savefig(os.path.join(output_path, plot_filename), dpi=300)
  #  plt.close(fig)


# Bar chart plotting section
model_labels = ['Astro-ResCBAM-UNet', 'Unet', 'Segformer', 'FPN']

# Get the accumulated values for each metric
tn_values = [test_tn_meter1.sum, test_tn_meter2.sum, test_tn_meter3.sum, test_tn_meter4.sum]
tp_values = [test_tp_meter1.sum, test_tp_meter2.sum, test_tp_meter3.sum, test_tp_meter4.sum]
fp_values = [test_fp_meter1.sum, test_fp_meter2.sum, test_fp_meter3.sum, test_fp_meter4.sum]
fn_values = [test_fn_meter1.sum, test_fn_meter2.sum, test_fn_meter3.sum, test_fn_meter4.sum]

# Create separate plots for each metric
metrics = [
    # True Negatives: Dự đoán đúng là nền (Background) - Thường dùng màu xanh lá đậm hoặc xám trung tính
    {'name': 'True Negatives (TN)', 'values': tn_values, 'color': '#2ecc71', 'filename': 'tn_bar_chart.png'}, # Emerald Green

    # True Positives: Dự đoán đúng là Tòa nhà (Building) - Màu xanh dương biểu thị sự tin cậy
    {'name': 'True Positives (TP)', 'values': tp_values, 'color': '#3498db', 'filename': 'tp_bar_chart.png'}, # Peter River Blue

    # False Positives: Lỗi báo giả (Nhầm nền thành nhà) - Màu đỏ để cảnh báo
    {'name': 'False Positives (FP)', 'values': fp_values, 'color': '#e74c3c', 'filename': 'fp_bar_chart.png'}, # Alizarin Red

    # False Negatives: Lỗi bỏ sót (Nhà nhưng không biết) - Màu cam để chỉ sự thiếu sót
    {'name': 'False Negatives (FN)', 'values': fn_values, 'color': '#f39c12', 'filename': 'fn_bar_chart.png'}, # Orange
]

# metrics = [
#    {'name': 'True Negatives (TN)', 'values': tn_values, 'color': 'green', 'filename': 'tn_bar_chart.png'},
#    {'name': 'True Positives (TP)', 'values': tp_values, 'color': 'blue', 'filename': 'tp_bar_chart.png'},
#    {'name': 'False Positives (FP)', 'values': fp_values, 'color': 'red', 'filename': 'fp_bar_chart.png'},
#    {'name': 'False Negatives (FN)', 'values': fn_values, 'color': 'orange', 'filename': 'fn_bar_chart.png'},
# ]

for metric in metrics:
   fig_bar, ax_bar = plt.subplots(1, 1, figsize=(10, 6))
   ax_bar.bar(model_labels, metric['values'], color=metric['color'], edgecolor='grey')
   ax_bar.set_title(f'Accumulated {metric["name"]} Comparison', fontweight='bold')
   ax_bar.set_xlabel('Models', fontweight='bold')
   ax_bar.set_ylabel('Accumulated Pixel Count', fontweight='bold')
   ax_bar.grid(axis='y', linestyle='--', alpha=0.7)
   plt.tight_layout(pad=3.0)

   # Save the file to the specified output path
   plt.savefig(os.path.join(output_path, metric['filename']), dpi=300)
   plt.close(fig_bar)


# Save accumulated metrics to a CSV file in the 'test' folder
df = pd.DataFrame({
   'Model': model_labels,
   'True Negatives (TN)': tn_values,
   'True Positives (TP)': tp_values,
   'False Positives (FP)': fp_values,
   'False Negatives (FN)': fn_values
})
df.to_csv(os.path.join(output_path, 'confusion_metrics.csv'), index=False)


# Save average metrics to a new CSV file, with values rounded to 3 decimal places.
data = {
   'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'IoU'],
   'Astro-ResCBAM-UNet': [test_acc_meter1.avg, test_precision_meter1.avg, test_recall_meter1.avg, test_F1Score_meter1.avg, test_iou_meter1.avg],
   'Unet': [test_acc_meter2.avg, test_precision_meter2.avg, test_recall_meter2.avg, test_F1Score_meter2.avg, test_iou_meter2.avg],
   'Segformer': [test_acc_meter3.avg, test_precision_meter3.avg, test_recall_meter3.avg, test_F1Score_meter3.avg, test_iou_meter3.avg],
   'FPN': [test_acc_meter4.avg, test_precision_meter4.avg, test_recall_meter4.avg, test_F1Score_meter4.avg, test_iou_meter4.avg]
}

df_metrics = pd.DataFrame(data)
df_metrics = df_metrics.round(3)

df_metrics.to_csv(os.path.join(output_path, 'test_metrics_summary.csv'), index=False)

# Data for the performance metrics table
metrics_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'IoU'],
    'Astro-ResCBAM-UNet': [test_acc_meter1.avg, test_precision_meter1.avg, test_recall_meter1.avg, test_F1Score_meter1.avg, test_iou_meter1.avg],
    'Unet': [test_acc_meter2.avg, test_precision_meter2.avg, test_recall_meter2.avg, test_F1Score_meter2.avg, test_iou_meter2.avg],
    'Segformer': [test_acc_meter3.avg, test_precision_meter3.avg, test_recall_meter3.avg, test_F1Score_meter3.avg, test_iou_meter3.avg],
    'FPN': [test_acc_meter4.avg, test_precision_meter4.avg, test_recall_meter4.avg, test_F1Score_meter4.avg, test_iou_meter4.avg]
}

# Data for the confusion matrix table
confusion_data = {
    'Metric': ['True Negatives (TN)', 'True Positives (TP)', 'False Negatives (FN)', 'False Positives (FP)'],
    'Astro-ResCBAM-UNet': [test_tn_meter1.sum, test_tp_meter1.sum, test_fn_meter1.sum, test_fp_meter1.sum],
    'Unet': [test_tn_meter2.sum, test_tp_meter2.sum, test_fn_meter2.sum, test_fp_meter2.sum],
    'Segformer': [test_tn_meter3.sum, test_tp_meter3.sum, test_fn_meter3.sum, test_fp_meter3.sum],
    'FPN': [test_tn_meter4.sum, test_tp_meter4.sum, test_fn_meter4.sum, test_fp_meter4.sum]
}

# Create DataFrames
metrics_df = pd.DataFrame(metrics_data)
confusion_df = pd.DataFrame(confusion_data)

# Set 'Metric' as the index for cleaner display
metrics_df.set_index('Metric', inplace=True)
confusion_df.set_index('Metric', inplace=True)

# Print the tables
print("### Performance Metrics Table")
print(metrics_df.to_string(float_format='%.3f'))
print("\n" + "="*50 + "\n")
print("### Accumulated Confusion Matrix Pixel Counts")
print(confusion_df.to_string())

Output hidden; open in https://colab.research.google.com to view.